# 1번 구조 기준선 비교 재현

**담당** 김대건. 아래 코드는 AI가 작성한 연구 보조 자료이며 개인이 실제 검토하고 실행한 범위를 별도로 기록한다.

**사례** 사전에 무작위로 고정한 10개 그래프와 두 예산 조건의 총 20개 사례를 사용한다. 기지 수는 8·12·16·24·32개이며 크기별 시드가 두 개다. n=8·12는 공개 문제은행에서 무작위로 고르고 n=16·24·32는 새 시드로 생성했다. 사례 목록과 무작위 추출법은 `results/agent-study/2026-11-24-sized/protocol.json`에 보관한다. 최적 참조는 n=8·12의 8개 사례에서 확인할 수 있다.

**재현** 프로젝트 루트에서 `pip install -e '.[test]'`로 설치한 후 아래 셀을 순서대로 실행한다. 전략별 실행은 최대 300초이며 공통 평가기는 제출된 방호안을 평가한다. 로컬 재현의 주소 공간 제한은 웹 cgroup 메모리 한도와 다르다.

**출처** 합성 그래프 생성 참고 [Dong·Tian (2022)](https://doi.org/10.1155/2022/3108327), 매개중심성 계산 참고 [Brandes (2001)](https://doi.org/10.1080/0022250X.2001.9990249). 데이터는 [프로젝트 저장소](https://github.com/d86729/os-project-26-2)의 기존 은행과 연구용 사례이며 확인일은 2026-11-24다.

### 담당 김대건 · 환경과 고정 입력 확인

In [1]:
from pathlib import Path
import hashlib
import json
import os
import resource
import subprocess
import sys

from protection.runner import initialize_single_thread, collect_environment
initialize_single_thread()
from protection.models import load_bundle
from protection.program import protected_ids, scenario_input
from protection.evaluation import Evaluator

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / 'pyproject.toml').is_file())
study = root / 'results/agent-study/2026-11-24-sized'
protocol = json.loads((study / 'protocol.json').read_text(encoding='utf-8'))
cases = json.loads((study / 'cases.json').read_text(encoding='utf-8'))
saved = [json.loads(line) for line in (study / 'runs.jsonl').read_text(encoding='utf-8').splitlines()]
methods = {item['method']: item for item in protocol['strategies']}
assert len(cases) == 20
environment = collect_environment()
print('운영체제:', environment['os'])
print('CPU:', environment['cpu'])
print('Python:', environment['python'])
print('사례:', len(cases), '· 그래프:', len({(c['n'], c['seed']) for c in cases}))

운영체제: Linux-6.18.58.1-microsoft-standard-WSL2-x86_64-with-glibc2.41
CPU: Intel(R) Core(TM) i5-8500 CPU @ 3.00GHz
Python: 3.13.5
사례: 20 · 그래프: 10


### 담당 김대건 · 두 전략 실행과 공식 평가

In [2]:
from collections import Counter
from IPython.display import Markdown, display

def limit_child():
    resource.setrlimit(resource.RLIMIT_AS, (6_000_000_000, 6_000_000_000))
    available = os.sched_getaffinity(0)
    os.sched_setaffinity(0, {min(available)})

outcomes = Counter()
lines = ['| 사례 | 매개중심성 연결 쌍 / 전달 점수 | 차수 연결 쌍 / 전달 점수 | 결과 | 최적 참조 |',
         '|---|---:|---:|---|---|']
for case in cases:
    bundle = load_bundle(root / case['path'], require_reference=case['reference_available'])
    assert bundle.case_id == case['case_id']
    stdin = scenario_input(bundle)
    assert hashlib.sha256(stdin).hexdigest() == case['stdin_sha256']
    scores = {}
    for name in ('degree', 'betweenness'):
        method = methods[name]
        source = root / method['path']
        assert hashlib.sha256(source.read_bytes()).hexdigest() == method['sha256']
        process = subprocess.run([sys.executable, '-I', str(source)], input=stdin,
                                 capture_output=True, timeout=300, check=True,
                                 preexec_fn=limit_child)
        chosen = protected_ids(process.stdout, bundle)
        score = Evaluator(bundle.scenario).evaluate(list(chosen), bundle.budget)
        previous = next(r for r in saved if r['name'] == case['name'] and r['method'] == name)
        assert previous['status'] == 'completed'
        assert (score.connected_pairs, score.delivery_units) == (previous['connected_pairs'],
                                                                  previous['delivery_units'])
        scores[name] = score
    a, b = scores['betweenness'], scores['degree']
    outcome = '매개중심성' if a.score > b.score else '차수' if a.score < b.score else '동점'
    outcomes[outcome] += 1
    lines.append(f"| {case['name']} | {a.connected_pairs} / {a.delivery_units} | "
                 f"{b.connected_pairs} / {b.delivery_units} | {outcome} | "
                 f"{'있음' if case['reference_available'] else '미확인'} |")
display(Markdown(chr(10).join(lines)))
print('매개중심성 승:', outcomes['매개중심성'], '동점:', outcomes['동점'],
      '차수 승:', outcomes['차수'])

/home/user/os-project-26-2/protection/flow.py:187: OptimizeWarning: Unrecognized options detected: {'threads': 1}. These will be passed to HiGHS verbatim.
  result = linprog(lp.c, A_ub=lp.G, b_ub=lp.b,


| 사례 | 매개중심성 연결 쌍 / 전달 점수 | 차수 연결 쌍 / 전달 점수 | 결과 | 최적 참조 |
|---|---:|---:|---|---|
| n8-seed20-p20 | 16 / 509259 | 15 / 740741 | 매개중심성 | 있음 |
| n8-seed20-p40 | 21 / 861111 | 21 / 861111 | 동점 | 있음 |
| n8-seed4-p20 | 21 / 450000 | 16 / 600000 | 매개중심성 | 있음 |
| n8-seed4-p40 | 28 / 730000 | 21 / 650000 | 매개중심성 | 있음 |
| n12-seed3-p20 | 55 / 488000 | 55 / 488000 | 동점 | 있음 |
| n12-seed3-p40 | 55 / 744000 | 55 / 488000 | 매개중심성 | 있음 |
| n12-seed5-p20 | 55 / 592357 | 55 / 528662 | 매개중심성 | 있음 |
| n12-seed5-p40 | 55 / 592357 | 55 / 592357 | 동점 | 있음 |
| n16-seed330015641-p20 | 105 / 660714 | 105 / 660714 | 동점 | 미확인 |
| n16-seed330015641-p40 | 105 / 758929 | 105 / 758929 | 동점 | 미확인 |
| n16-seed1423693424-p20 | 79 / 743119 | 79 / 743119 | 동점 | 미확인 |
| n16-seed1423693424-p40 | 105 / 715596 | 105 / 743119 | 차수 | 미확인 |
| n24-seed738938632-p20 | 190 / 526984 | 210 / 615873 | 차수 | 미확인 |
| n24-seed738938632-p40 | 231 / 742857 | 231 / 701587 | 매개중심성 | 미확인 |
| n24-seed1598601602-p20 | 211 / 803333 | 211 / 803333 | 동점 | 미확인 |
| n24-seed1598601602-p40 | 231 / 923333 | 231 / 793333 | 매개중심성 | 미확인 |
| n32-seed1234542658-p20 | 352 / 672249 | 406 / 650718 | 차수 | 미확인 |
| n32-seed1234542658-p40 | 435 / 825359 | 406 / 650718 | 매개중심성 | 미확인 |
| n32-seed1370353568-p20 | 407 / 758621 | 406 / 652520 | 매개중심성 | 미확인 |
| n32-seed1370353568-p40 | 436 / 771883 | 435 / 748011 | 매개중심성 | 미확인 |

매개중심성 승: 10 동점: 7 차수 승: 3


## 해석

매개중심성 10승, 동점 7건, 차수 3승은 10개 그래프에서 두 예산을 적용한 기술적 결과다. n=16·24·32의 12개 사례는 완전탐색 참조가 아직 없으므로 두 방호안의 상대 점수만 비교한다. 크기별 그래프가 2개이므로 보편적 우열을 단정하기 어렵다. 저장된 원자료와 실행 환경은 `results/agent-study/2026-11-24-sized/`에서 확인한다.